# K-means segmentation of 500 homes

**Purpose:** group the 500 homes by their appliance-level consumption and test whether the groups are meaningful.

**Data:** Smart Home Energy Consumption (Kaggle), 100,000 readings, 500 homes, 10 appliance types. The data file is not stored in this repository; download it from Kaggle and set the path in the first code cell.

**Requires:** pandas, numpy, matplotlib, scikit-learn.

**Criteria for a meaningful segment**:
1. clearer than groups formed on shuffled data
2. stable when the analysis is repeated
3. describable in plain language

**Summary of findings:** the k-means groups are weakly separated (silhouette about 0.08), are not clearly better than groups formed on shuffled data, and are only marginally stable. We therefore use k = 3 as a working description, not as distinct household types.

## 1. Home-level feature table
- One row per home (500 rows), one column per appliance (10 columns).
- Each value is the mean kWh per reading, not the total, because homes have between 157 and 245 readings and totals would reward homes that were measured more often.
- Descriptive columns (`n_readings`, `avg_kwh`) are kept but not used for clustering.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/smart_home_energy_consumption_large.csv")

# Mean kWh per appliance for each home (500 rows x 10 columns)
features = df.pivot_table(
    index="Home ID",
    columns="Appliance Type",
    values="Energy Consumption (kWh)",
    aggfunc="mean"
)

print(features.shape)          # expect (500, 10)
print(features.isna().sum().sum(), "missing values")
features.head()

(500, 10)
0 missing values


Appliance Type,Air Conditioning,Computer,Dishwasher,Fridge,Heater,Lights,Microwave,Oven,TV,Washing Machine
Home ID,,,,,,,,,,
1,3.378261,1.144348,1.007222,0.274000,3.467727,1.237500,1.197222,1.115385,0.867222,1.301304
2,3.341765,1.002308,1.231667,0.301765,3.221200,1.076154,1.032500,1.237037,1.138333,1.210000
3,3.652500,1.063913,1.060000,0.318500,3.826667,1.148824,1.128857,1.282143,0.977500,1.195238
4,3.365000,1.171667,1.131667,0.351053,3.242778,1.001667,1.034615,0.837647,1.080952,1.097917
5,3.585789,1.143200,1.185263,0.340000,3.278636,1.035238,1.020000,1.258095,0.747143,1.065000


In [2]:
# Extra descriptive columns (not used for clustering)
features["n_readings"] = df.groupby("Home ID").size()
features["avg_kwh"] = df.groupby("Home ID")["Energy Consumption (kWh)"].mean()

# Save for later use (dashboard, report)
features.to_csv("../data/processed/home_features.csv")
features.shape

(500, 12)

## 2. K-means for k = 2 to 8
- Features are standardized (mean 0, std 1) so that large appliances (Air Conditioning, Heater, about 3.5 kWh) do not dominate small ones (Fridge, about 0.3 kWh).
- Silhouette score: 1 means well-separated clusters, 0 means overlapping, and below 0.25 is usually read as little structure.

In [3]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Keep only the 10 appliance columns
appliance_cols = [c for c in features.columns if c not in ["n_readings", "avg_kwh"]]

# Standardize: each column gets mean 0 and std 1
X = StandardScaler().fit_transform(features[appliance_cols])

results = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = km.fit_predict(X)
    results.append({
        "k": k,
        "silhouette": round(silhouette_score(X, labels), 3),
        "inertia": round(km.inertia_, 1)
    })

pd.DataFrame(results)

,k,silhouette,inertia
0,2,0.080,4589.8
1,3,0.076,4295.6
2,4,0.075,4075.1
3,5,0.078,3891.6
4,6,0.076,3773.7
5,7,0.076,3646.0
6,8,0.079,3535.1


**Result**
- Silhouette is 0.075 to 0.080 for every k from 2 to 8, so no k is clearly better.
- Inertia falls steadily from 4589.8 (k = 2) to 3535.1 (k = 8) with no elbow.
- All silhouette values are far below 0.25, so clusters overlap almost completely.

**Choice of k**
- Silhouette is flat across k, so k = 3 is chosen for balanced group sizes (184 / 175 / 141 homes), interpretability and higher stability than k = 5 (section 4), not because the data show an elbow.

## 3. Baseline: clustering on shuffled data
- Home IDs are shuffled, the 500 x 10 table is rebuilt, and k-means is repeated, to see what silhouette random data produce.
- If the real score is not above the shuffled range, the clusters are not meaningful.

In [4]:
rng = np.random.default_rng(42)
ks = [2,3,4,5]
n_shuffles = 30

def silhouette_for_k(data, k):
    labels = KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(data)
    return silhouette_score(data, labels)

# Real data
real_scores = {k: silhouette_for_k(X, k) for k in ks}

# Shuffled data: rebuild the feature table after shuffling Home ID
shuffled_scores = {k: [] for k in ks}
for _ in range(n_shuffles):
    shuffled_ids = rng.permutation(df["Home ID"].values)
    f = (df.assign(shuffled_home=shuffled_ids)
           .pivot_table(index="shuffled_home",
                        columns="Appliance Type",
                        values="Energy Consumption (kWh)",
                        aggfunc="mean"))
    Xs = StandardScaler().fit_transform(f)
    for k in ks:
        shuffled_scores[k].append(silhouette_for_k(Xs, k))

# Summary table
rows = []
for k in ks:
    p95 = np.percentile(shuffled_scores[k], 95)
    rows.append({
        "k": k,
        "real": round(real_scores[k], 3),
        "shuffled_mean": round(np.mean(shuffled_scores[k]), 3),
        "shuffled_p95": round(p95, 3),
        "above_p95": real_scores[k] > p95
    })

pd.DataFrame(rows)

,k,real,shuffled_mean,shuffled_p95,above_p95
0,2,0.080,0.078,0.081,False
1,3,0.076,0.073,0.076,False
2,4,0.075,0.073,0.076,False
3,5,0.078,0.074,0.078,True


In [5]:
# Follow-up for k = 5: 200 shuffles (new variable names, so the 30-shuffle results are kept)
rng_200 = np.random.default_rng(42)
real_k5 = silhouette_for_k(X, 5)
shuffled_k5 = []

for _ in range(200):
    ids = rng_200.permutation(df["Home ID"].values)
    f = (df.assign(shuffled_home=ids)
           .pivot_table(index="shuffled_home", columns="Appliance Type",
                        values="Energy Consumption (kWh)", aggfunc="mean"))
    Xs = StandardScaler().fit_transform(f)
    shuffled_k5.append(silhouette_for_k(Xs, 5))

print("Done:", len(shuffled_k5), "shuffles")

Done: 200 shuffles


In [6]:
# Exact comparison for k = 5 (200 shuffles)
scores = np.array(shuffled_k5)
real = real_k5

print("Real silhouette:", round(real, 5))
print("Shuffled mean:", round(scores.mean(), 5))
print("Shuffled std:", round(scores.std(), 5))
print("Shuffled max:", round(scores.max(), 5))
print("Real is", round((real - scores.mean()) / scores.std(), 2), "std above shuffled mean")

p_value = (np.sum(scores >= real) + 1) / (len(scores) + 1)
print("Empirical p-value:", round(p_value, 3))

print("Shuffled runs at least as high as real:", int(np.sum(scores >= real)), "of", len(scores))

Real silhouette: 0.07812
Shuffled mean: 0.07388
Shuffled std: 0.00228
Shuffled max: 0.08206
Real is 1.86 std above shuffled mean
Empirical p-value: 0.05
Shuffled runs at least as high as real: 9 of 200


**Result (30 shuffles, k = 2 to 5)**

- Real silhouette: 0.075 to 0.080 for k = 2 to 5; shuffled mean: 0.073 to 0.078.
- For k = 2, 3 and 4 the real score is not above the 95th percentile of shuffled data.
- For k = 5 it is marginally above the 95th percentile (0.078 vs 0.078 at three decimals). With four values of k tested, one marginal result can occur by chance.
- All scores are far below 0.25, the usual threshold for weak cluster structure.

**Follow-up for k = 5 (200 shuffles)**

- Real silhouette 0.0781 vs shuffled mean 0.0739 (std 0.0023); real is 1.86 std above the shuffled mean; empirical p-value 0.05.
- 9 of 200 shuffled runs scored at least as high as the real data (shuffled max 0.0821).
- Four values of k were tested, so the corrected p-value is about 0.20 (Bonferroni).

**Conclusion:** k-means clusters on this dataset are not distinguishable from clusters formed on random data. No evidence of meaningful segments.

## 4. Stability across random seeds
- K-means starts from random cluster centres, so different seeds can give different groupings.
- k-means is rerun with 10 different seeds; the adjusted Rand index (ARI) measures how much the grouping changes (1 = identical, 0 = no better than random).
- The same test is then repeated on shuffled data to see how stable random data are. A stability score only means something when compared with that baseline.

### 4a: Stability of the real data (store the results for later cells)

In [7]:
from sklearn.metrics import adjusted_rand_score

real_ari = {}
for k in [3, 5]:
    base = KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(X)
    ari_scores = []
    for seed in range(1, 11):
        other = KMeans(n_clusters=k, n_init=10, random_state=seed).fit_predict(X)
        ari_scores.append(adjusted_rand_score(base, other))
    real_ari[k] = np.mean(ari_scores)
    print(f"k = {k}: mean ARI = {np.mean(ari_scores):.3f}, min = {np.min(ari_scores):.3f}")

k = 3: mean ARI = 0.696, min = 0.323
k = 5: mean ARI = 0.433, min = 0.306


**Result for the real data**

| k | mean ARI | minimum ARI |
|---|---|---|
| 3 | 0.696 | 0.323 |
| 5 | 0.433 | 0.306 |

- k = 5 changes substantially between seeds; k = 3 is moderately stable.
- The minimum shows that at least one seed gave a noticeably different grouping for both values of k.

### 4b. Stability baseline: one shuffled dataset
- Home IDs are shuffled once, the 500 x 10 table is rebuilt, and the same seed test is run.
- This is a quick check; one shuffle cannot show how much random data vary.

In [8]:
rng = np.random.default_rng(7)
shuffled_ids = rng.permutation(df["Home ID"].values)
f = (df.assign(shuffled_home=shuffled_ids)
       .pivot_table(index="shuffled_home", columns="Appliance Type",
                    values="Energy Consumption (kWh)", aggfunc="mean"))
Xs = StandardScaler().fit_transform(f)

for k in [3, 5]:
    base = KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(Xs)
    ari_scores = [adjusted_rand_score(base,
                  KMeans(n_clusters=k, n_init=10, random_state=s).fit_predict(Xs))
                  for s in range(1, 11)]
    print(f"Shuffled, k = {k}: mean ARI = {np.mean(ari_scores):.3f}, min = {np.min(ari_scores):.3f}")

Shuffled, k = 3: mean ARI = 0.507, min = 0.290
Shuffled, k = 5: mean ARI = 0.284, min = 0.122


**Result (one shuffle):** 
- k = 3: mean ARI 0.507 (min 0.290); k = 5: mean ARI 0.284 (min 0.122).
- Shuffled data are less stable than the real data for both k, but one run does not show how much shuffled results vary. A larger number of shuffles is needed.

### 4c. Stability baseline: 100 shuffled datasets
- The shuffle and the seed test are repeated on 100 shuffled datasets.
- For each dataset the mean ARI over 10 seeds is stored, which gives the range of stability that random data produce.

In [9]:
rng = np.random.default_rng(123)
shuffled_ari = {3: [], 5: []}

for _ in range(100):
    ids = rng.permutation(df["Home ID"].values)
    f = (df.assign(shuffled_home=ids)
           .pivot_table(index="shuffled_home", columns="Appliance Type",
                        values="Energy Consumption (kWh)", aggfunc="mean"))
    Xs = StandardScaler().fit_transform(f)
    for k in [3, 5]:
        base = KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(Xs)
        aris = [adjusted_rand_score(base,
                KMeans(n_clusters=k, n_init=10, random_state=s).fit_predict(Xs))
                for s in range(1, 11)]
        shuffled_ari[k].append(np.mean(aris))


for k in [3, 5]:
    arr = np.array(shuffled_ari[k])
    print(f"k = {k}: real {real_ari[k]:.3f} | shuffled mean {arr.mean():.3f}, "
          f"std {arr.std():.3f}, max {arr.max():.3f}")

k = 3: real 0.696 | shuffled mean 0.414, std 0.140, max 0.822
k = 5: real 0.433 | shuffled mean 0.283, std 0.057, max 0.481


**Result (100 shuffles)**

| k | real ARI | shuffled mean | shuffled std | shuffled max |
|---|---|---|---|---|
| 3 | 0.696 | 0.414 | 0.140 | 0.822 |
| 5 | 0.433 | 0.283 | 0.057 | 0.481 |

- The real data are more stable than the average shuffled dataset, but the best shuffled datasets are more stable than the real data at both k (0.822 and 0.481).
- An earlier run with 20 shuffles suggested the real data beat all shuffled runs; this did not hold with 100 shuffles, so small numbers of shuffles should not be used for conclusions here.

### 4d. Empirical p-values
- For each k, count how many shuffled datasets were at least as stable as the real data.
- Empirical p-values (counting runs) are used rather than standard deviations, because the shuffled ARI distribution is skewed to the right.

In [10]:
for k in [3, 5]:
    arr = np.array(shuffled_ari[k])
    p = (np.sum(arr >= real_ari[k]) + 1) / (len(arr) + 1)
    print(f"k = {k}: runs at least as stable as real: {np.sum(arr >= real_ari[k])} of {len(arr)}, p = {p:.3f}")

k = 3: runs at least as stable as real: 1 of 100, p = 0.020
k = 5: runs at least as stable as real: 2 of 100, p = 0.030


**Result**

| k | shuffled runs at least as stable as real | p-value | p x 2 (two values of k) |
|---|---|---|---|
| 3 | 1 of 100 | 0.020 | 0.04 |
| 5 | 2 of 100 | 0.030 | 0.06 |

**Conclusion:** the real data are marginally more stable than shuffled data. After correcting for two values of k, k = 3 stays just below 0.05 and k = 5 does not. This marginal result is not replicated by Hierarchical clustering (p = 0.455) or Gaussian Mixture (p = 0.921), so we treat it as a faint hint, not as evidence of real segments.

## 5. Profile of the k = 3 segments
- Mean kWh per appliance in each cluster, and the difference from the overall mean in standard deviations.

In [11]:
km3 = KMeans(n_clusters=3, n_init=10, random_state=0).fit(X)
labels3 = pd.Series(km3.labels_, index=features.index, name="cluster")

# Mean kWh per appliance in each cluster (original units)
profile = features[appliance_cols].groupby(labels3).mean().round(2)
profile["n_homes"] = labels3.value_counts().sort_index()
print(profile)

# Difference from the overall mean, in standard deviations
z = pd.DataFrame(X, index=features.index, columns=appliance_cols)
print(z.groupby(labels3).mean().round(2))

Appliance Type  Air Conditioning  Computer  Dishwasher  Fridge  Heater  \
cluster                                                                  
0                           3.61      1.06        1.15    0.30    3.46   
1                           3.45      1.11        1.10    0.29    3.50   
2                           3.40      1.13        1.04    0.32    3.52   

Appliance Type  Lights  Microwave  Oven    TV  Washing Machine  n_homes  
cluster                                                                  
0                 1.09       1.13  1.12  1.14             1.16      184  
1                 1.04       1.10  1.05  1.03             1.02      175  
2                 1.14       1.06  1.16  1.11             1.12      141  
         Air Conditioning  Computer  Dishwasher  Fridge  Heater  Lights  \
cluster                                                                   
0                    0.62     -0.33        0.38   -0.04   -0.14    0.02   
1                   -0.25      0.1

**Result**
- Segment sizes: 184, 175 and 141 homes.
- Differences between segments are small in kWh: Air Conditioning varies by about 0.2 kWh (3.40 to 3.61), most other appliances by 0.05 to 0.15 kWh.
- Segment 0: higher Air Conditioning, Washing Machine, TV. Segment 1: lower Washing Machine, Oven, TV, Fridge, Lights. Segment 2: higher Fridge, Lights, Oven; lower Air Conditioning, Dishwasher.
- Labels (0, 1, 2) are arbitrary and can change with the seed; they are fixed with `random_state = 0` and saved to `home_segments.csv`.
- Names are descriptive only and do not imply behaviour.

In [12]:
# Save cluster labels so the whole team and the dashboard use the same segmentation
segments = features.copy()
segments["cluster"] = labels3
segments["segment_name"] = segments["cluster"].map({
    0: "AC and laundry-leaning",
    1: "Lower appliance use",
    2: "Fridge, lights and oven-leaning"
})
segments.to_csv("../data/processed/home_segments.csv")
segments["segment_name"].value_counts()

segment_name
AC and laundry-leaning             184
Lower appliance use                175
Fridge, lights and oven-leaning    141
Name: count, dtype: int64

## Conclusion and limitations
- No reliable evidence of meaningful household segments in this dataset: silhouette about 0.08, not above shuffled data (k = 2 to 4), and stability only marginal.
- k = 3 is used as a working description because it gives balanced groups and is more stable than k = 5, not because the data show an elbow (silhouette is flat across k).
- The dataset appears synthetic (no seasonality, household size varies within every home), so these findings may not apply to real data.
- Recommended primary comparison: percentile of each home against all 500 homes, per appliance; segments are a secondary view.